# Dashboard: Surveillance (Phase 7)

This dashboard subscribes to live MQTT streams and renders city surveillance state:
- humans movement/state
- illegal events
- camera detections
- active alarms

Visual precedence:
1. Base color = deterministic unique per human
2. Black when latest detection for human is `detected=false`
3. Red alarm blink is a separate marker for one step (`ttl_steps=1`)

In [20]:
from __future__ import annotations

import colorsys
import json
import math
import threading
import time
from collections import deque
from typing import Any

from IPython.display import display
from anymap_ts import Map
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

In [21]:
# Config + constants
cfg = load_config()
sim_cfg = cfg.simulation

PARKEN_LAT = 55.7029
PARKEN_LON = 12.5720

SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_HUMANS_STATE = f"{SURVEILLANCE_TOPIC_ROOT}/humans/state"
TOPIC_EVENTS_ILLEGAL = f"{SURVEILLANCE_TOPIC_ROOT}/events/illegal"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_ALARMS = f"{SURVEILLANCE_TOPIC_ROOT}/alarms/active"

LIVE_QOS = 0
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0

print(f"Dashboard broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> {TOPIC_HUMANS_STATE}, {TOPIC_EVENTS_ILLEGAL}, {TOPIC_DETECTIONS}, {TOPIC_ALARMS}")
print(f"Dashboard step delay: {STEP_DELAY_S}s")

Dashboard broker: 127.0.0.1:1883
Topics -> simcity/surveillance/humans/state, simcity/surveillance/events/illegal, simcity/surveillance/detections/camera, simcity/surveillance/alarms/active
Dashboard step delay: 1.0s


In [22]:
# Coordinate transform + deterministic colors
def local_xy_to_lnglat(x_m: float, y_m: float, center_lat: float, center_lon: float) -> tuple[float, float]:
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = 111_320.0 * math.cos(math.radians(center_lat))
    lat = center_lat + (y_m / meters_per_deg_lat)
    lon = center_lon + (x_m / meters_per_deg_lon)
    return lon, lat

def stable_color_for_human(human_id: str) -> str:
    hue = (sum(ord(ch) for ch in human_id) % 360) / 360.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

In [23]:
# Map + dashboard state
m = Map(center=(PARKEN_LON, PARKEN_LAT), zoom=15.8, height="700px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")
display(m)

state_lock = threading.Lock()
inbox: deque[dict[str, Any]] = deque()

humans: dict[str, dict[str, Any]] = {}
events_by_id: dict[str, dict[str, Any]] = {}
latest_detection_by_human: dict[str, bool] = {}
active_alarms: dict[str, dict[str, Any]] = {}
criminal_acts_by_human: dict[str, int] = {}
scanned_human_ids: set[str] = set()
processed_event_ids: set[str] = set()

known_human_markers: set[str] = set()
known_alarm_markers: set[str] = set()

anchor_step: int | None = None
anchor_started_at: float | None = None

stats = {
    "humans_state": 0,
    "events_illegal": 0,
    "detections": 0,
    "alarms": 0,
    "dropped_invalid": 0,
}

# Diagnostic counters for MQTT callback visibility
callback_total = 0
callback_by_topic: dict[str, int] = {}
last_callback_topic: str | None = None

mqtt_connector: MqttConnector | None = None
dashboard_connected = False

def current_local_step_unlocked() -> int | None:
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)

print("Dashboard map initialized.")

Dashboard map initialized.


In [27]:
# MQTT callback ingestion (thread-safe queue only; no UI updates here)
TOPIC_KIND_MAP = {
    TOPIC_HUMANS_STATE: "humans_state",
    TOPIC_EVENTS_ILLEGAL: "events_illegal",
    TOPIC_DETECTIONS: "detections",
    TOPIC_ALARMS: "alarms",
}

def enqueue_message(kind: str, payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at

    with state_lock:
        if anchor_step is None and "step" in payload:
            try:
                anchor_step = int(payload["step"])
                anchor_started_at = time.perf_counter()
            except Exception:
                pass

        inbox.append({"kind": kind, "payload": payload})

def on_any_message(client, userdata, message):
    global callback_total, last_callback_topic

    with state_lock:
        callback_total += 1
        callback_by_topic[message.topic] = callback_by_topic.get(message.topic, 0) + 1
        last_callback_topic = message.topic

    kind = TOPIC_KIND_MAP.get(message.topic)
    if kind is None:
        return

    try:
        payload = json.loads(message.payload.decode("utf-8"))
        enqueue_message(kind, payload)
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1

try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="dashboard")
    mqtt_connector.connect()

    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_connector.client.on_message = on_any_message

        mqtt_connector.client.subscribe(TOPIC_HUMANS_STATE, qos=LIVE_QOS)
        mqtt_connector.client.subscribe(TOPIC_EVENTS_ILLEGAL, qos=LIVE_QOS)
        mqtt_connector.client.subscribe(TOPIC_DETECTIONS, qos=LIVE_QOS)
        mqtt_connector.client.subscribe(TOPIC_ALARMS, qos=LIVE_QOS)

        dashboard_connected = True
        print("Dashboard subscribed to all surveillance topics.")
    else:
        print("Dashboard MQTT timeout. Running in idle mode.")
except Exception as exc:
    print(f"Dashboard MQTT unavailable ({exc}). Running in idle mode.")

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Dashboard subscribed to all surveillance topics.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [ ]:
# Render loop (updates map from queued messages)
def apply_inbox_messages() -> None:
    while True:
        with state_lock:
            if not inbox:
                break
            item = inbox.popleft()

        kind = item["kind"]
        payload = item["payload"]

        if kind == "humans_state":
            try:
                human_id = str(payload["human_id"])
                humans[human_id] = {
                    "name": str(payload.get("name", human_id)),
                    "x": float(payload["x"]),
                    "y": float(payload["y"]),
                    "step": int(payload["step"]),
                }
                with state_lock:
                    stats["humans_state"] += 1
            except Exception:
                with state_lock:
                    stats["dropped_invalid"] += 1

        elif kind == "events_illegal":
            try:
                event_id = str(payload["event_id"])
                human_id = str(payload["human_id"])
                is_new_event = event_id not in processed_event_ids
                events_by_id[event_id] = {
                    "step": int(payload["step"]),
                    "human_id": human_id,
                    "x": float(payload["x"]),
                    "y": float(payload["y"]),
                }
                if is_new_event:
                    processed_event_ids.add(event_id)
                    criminal_acts_by_human[human_id] = criminal_acts_by_human.get(human_id, 0) + 1
                with state_lock:
                    stats["events_illegal"] += 1
            except Exception:
                with state_lock:
                    stats["dropped_invalid"] += 1

        elif kind == "detections":
            try:
                human_id = str(payload["human_id"])
                detected = bool(payload["detected"])
                latest_detection_by_human[human_id] = detected
                scanned_human_ids.add(human_id)
                with state_lock:
                    stats["detections"] += 1
            except Exception:
                with state_lock:
                    stats["dropped_invalid"] += 1

        elif kind == "alarms":
            try:
                event_id = str(payload["event_id"])
                step = int(payload["step"])
                ttl_steps = int(payload.get("ttl_steps", 1))
                x = float(payload["x"])
                y = float(payload["y"])
                active_alarms[event_id] = {
                    "x": x,
                    "y": y,
                    "until_step": step + ttl_steps,
                }
                with state_lock:
                    stats["alarms"] += 1
            except Exception:
                with state_lock:
                    stats["dropped_invalid"] += 1

def render_dashboard_once() -> None:
    apply_inbox_messages()

    local_step = current_local_step_unlocked()

    for human_id, state in humans.items():
        base_color = stable_color_for_human(human_id)
        marker_color = "#000000" if latest_detection_by_human.get(human_id) is False else base_color

        lng, lat = local_xy_to_lnglat(state["x"] - 500.0, state["y"] - 500.0, PARKEN_LAT, PARKEN_LON)

        if human_id in known_human_markers:
            m.remove_marker(human_id)
        m.add_marker(lng, lat, name=human_id, color=marker_color, popup=state["name"])
        known_human_markers.add(human_id)

    expired_alarm_ids: list[str] = []
    for event_id, alarm in active_alarms.items():
        if local_step is not None and local_step >= int(alarm["until_step"]):
            expired_alarm_ids.append(event_id)
            continue

        alarm_name = f"alarm:{event_id}"
        lng, lat = local_xy_to_lnglat(alarm["x"] - 500.0, alarm["y"] - 500.0, PARKEN_LAT, PARKEN_LON)

        if alarm_name in known_alarm_markers:
            m.remove_marker(alarm_name)
        m.add_marker(lng, lat, name=alarm_name, color="#ff0000", popup=f"ALARM {event_id}")
        known_alarm_markers.add(alarm_name)

    for event_id in expired_alarm_ids:
        alarm_name = f"alarm:{event_id}"
        if alarm_name in known_alarm_markers:
            m.remove_marker(alarm_name)
            known_alarm_markers.remove(alarm_name)
        active_alarms.pop(event_id, None)

def run_dashboard_loop(total_steps: int = 60) -> None:
    print("Dashboard render loop started...")
    for _ in range(total_steps):
        loop_started = time.perf_counter()
        render_dashboard_once()
        elapsed = time.perf_counter() - loop_started
        time.sleep(max(0.0, STEP_DELAY_S - elapsed))
    print("Dashboard render loop complete.")

In [25]:
# Start dashboard loop (run while producer + detector are active)
run_dashboard_loop(total_steps=60)

Dashboard render loop started...


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

Dashboard render loop complete.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [ ]:
# Investigation: transform sanity + counters + active alarms + scanned humans summary
transform_samples = [
    (0.0, 0.0),
    (500.0, 500.0),
    (1000.0, 1000.0),
    (250.0, 750.0),
]

print("Transform sanity samples (local meters -> lng/lat):")
for x, y in transform_samples:
    lng, lat = local_xy_to_lnglat(x - 500.0, y - 500.0, PARKEN_LAT, PARKEN_LON)
    print(f"- x={x:.1f}, y={y:.1f} -> lng={lng:.6f}, lat={lat:.6f}")

with state_lock:
    local_step_snapshot = current_local_step_unlocked()
    stats_snapshot = dict(stats)
    callback_total_snapshot = callback_total
    callback_by_topic_snapshot = dict(callback_by_topic)
    last_callback_topic_snapshot = last_callback_topic
    inbox_depth_snapshot = len(inbox)

print("\nDashboard counters:")
print(f"- local_current_step={local_step_snapshot}")
for key, value in stats_snapshot.items():
    print(f"- {key}={value}")
print(f"- callback_total={callback_total_snapshot}")
print(f"- last_callback_topic={last_callback_topic_snapshot}")
print(f"- inbox_depth={inbox_depth_snapshot}")

if callback_by_topic_snapshot:
    print("- callback_by_topic:")
    for topic, count in callback_by_topic_snapshot.items():
        print(f"  - {topic}: {count}")

print("\nScanned humans + criminal acts:")
if scanned_human_ids:
    for human_id in sorted(scanned_human_ids):
        human_name = humans.get(human_id, {}).get("name", human_id)
        criminal_count = criminal_acts_by_human.get(human_id, 0)
        print(f"- {human_name}: criminal_acts={criminal_count}")
else:
    print("- none")

print("\nActive alarms (event_id -> until_step):")
if active_alarms:
    for event_id, alarm in active_alarms.items():
        print(f"- {event_id}: until_step={alarm['until_step']}")
else:
    print("- none")

Transform sanity samples (local meters -> lng/lat):
- x=0.0, y=0.0 -> lng=12.564029, lat=55.698408
- x=500.0, y=500.0 -> lng=12.572000, lat=55.702900
- x=1000.0, y=1000.0 -> lng=12.579971, lat=55.707392
- x=250.0, y=750.0 -> lng=12.568014, lat=55.705146

Dashboard counters:
- local_current_step=None
- humans_state=0
- events_illegal=0
- detections=0
- alarms=0
- dropped_invalid=0
- callback_total=0
- last_callback_topic=None
- inbox_depth=0

Scanned humans + criminal acts:
- none

Active alarms (event_id -> until_step):
- none


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [9]:
# Cleanup
if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()
    print("Dashboard MQTT connector disconnected.")
else:
    print("Dashboard cleanup: no active MQTT connection.")

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Dashboard MQTT connector disconnected.
